# Synthesize a Relational Database from a SQL Schema with PluRel

This notebook shows the minimal end-to-end flow:

1. Write a small `schema.sql` file (two tables + FK constraint).
2. Configure PluRel synthesis parameters.
3. Generate a synthetic database.
4. Inspect the resulting tables.

**Note:**
- This notebook assumes `plurel` is installed and importable in your environment.
- text columns (other than enums) are not supported yet.

## Prerequisites

In [1]:
# Ensure proper imports

## 1. Create `schema.sql`

In [2]:
from pathlib import Path

schema_sql = """CREATE TABLE users (
    user_id BIGINT PRIMARY KEY,
    status TEXT CHECK (status IN ('active', 'inactive', 'banned'))
);

CREATE TABLE orders (
    order_id BIGINT PRIMARY KEY,
    user_id BIGINT,
    amount DOUBLE,
    order_type TEXT CHECK (order_type IN ('online', 'instore')),
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
"""

schema_path = Path("schema.sql")
schema_path.write_text(schema_sql)
print(f"Wrote schema to: {schema_path.resolve()}")

Wrote schema to: /home/lzd/workspace/myplurel/mysyn/examples/schema.sql


## 2. Configure synthesis

In [4]:
from plurel import Choices, Config, DatabaseParams, SyntheticDataset

config = Config(
    database_params=DatabaseParams(
        num_rows_entity_table_choices=Choices("range", [50, 100]),
        num_rows_activity_table_choices=Choices("range", [500, 1000]),
    ),
    schema_file=str(schema_path),  # pass the schema file to config
)

print(config)

/home/lzd/workspace/myplurel/mysyn/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config(database_params=DatabaseParams(table_layout_choices=Choices(kind='set', value=['BarabasiAlbert', 'ReverseRandomTree', 'WattsStrogatz']), num_tables_choices=Choices(kind='range', value=[3, 20]), num_rows_entity_table_choices=Choices(kind='range', value=[50, 100]), num_rows_activity_table_choices=Choices(kind='range', value=[500, 1000]), num_cols_choices=Choices(kind='range', value=[3, 40]), min_timestamp=Timestamp('1990-01-01 00:00:00'), max_timestamp=Timestamp('2025-01-01 00:00:00'), column_nan_perc_choices=Choices(kind='range', value=[0.01, 0.1]), col_transform_choices=Choices(kind='set', value=['identity', 'rank_uniform', 'log', 'sqrt', 'standardize'])), scm_params=SCMParams(scm_layout_choices=Choices(kind='set', value=['ErdosRenyi', 'BarabasiAlbert', 'RandomTree', 'ReverseRandomTree', 'Layered', 'WattsStrogatz', 'RandomCauchy']), scm_col_node_perc_choices=Choices(kind='range', value=[0.3, 0.9]), num_categories_choices=Choices(kind='range', value=[2, 10]), col_stype_choices=Ch

## 3. Generate one synthetic database

In [5]:
db = SyntheticDataset(seed=0, config=config).make_db()
db

Database()

## 4. Inspect generated tables

In [6]:
# Tables are returned as a mapping: table_name -> pandas.DataFrame (or equivalent table object)
tables = db.table_dict
list(tables.keys())

['orders', 'users']

In [7]:
# preview of each table
for name, table in tables.items():
    print(f"\n=== {name} ===")
    print(table.df.head())


=== orders ===
   order_id  user_id    amount order_type                       date
0         0       25 -2.399767    instore 2014-09-08 00:00:00.000000
1         1        3 -2.399864    instore 2014-09-09 17:27:01.978021
2         2       23 -2.399808    instore 2014-09-11 10:54:03.956043
3         3        3 -2.401373    instore 2014-09-13 04:21:05.934065
4         4        4 -2.401317    instore 2014-09-14 21:48:07.912087

=== users ===
   user_id    status
0        0  inactive
1        1  inactive
2        2  inactive
3        3  inactive
4        4  inactive
